# Pathway-constrained variational inference on Kang PBMCs

One model, trained on the full gene panel, examined closely.

The model is an informed VAE: a Reactome gene-to-pathway map is imposed as a binary
mask on the encoder's first linear layer, so each latent unit reads only the genes
of one pathway. Training uses a negative-binomial likelihood on raw counts, with
log1p CP10K expression as the encoder's input and condition plus cell type threaded
through as covariates. The dataset is Kang et al. (2018): human PBMCs, control
against interferon-β stimulated, which makes interferon signalling a known right
answer to score any ranking against.

**What this notebook is.** A close look at one configuration, chosen by a sweep, with
the figures that show whether its pathway units mean anything.

**What it is not.** The evidence for the configuration. That lives in
`02_architecture_ablation.py`, `03_sample_size_experiment.py` and
`04_encoder_fidelity_experiment.py`, run as a Snakemake grid of 28 jobs whose merged
results sit in `outputs/*.csv`. Where a claim here rests on the sweep, the notebook
shows one figure and names the CSV. It does not re-derive an underpowered version of
a comparison the grid already settles.

> **For readers new to this.** A plain VAE learns whatever latent axes minimise
> reconstruction error, and those axes mean nothing in particular. Masking the first
> layer to a pathway map buys you axes that are named in advance: unit *j* is
> "Reactome interferon alpha/beta signalling". The whole question this notebook asks
> is whether that name is honest, because the mask constrains which genes a unit may
> read and nothing at all about what it does with them.

## Running it

    pixi run -e cuda12 jupyter lab        # from the repository root

Training the full panel takes about 4 minutes on an A100, and the whole notebook
about 5. The trainer moves the expression matrices onto the device once and indexes
batches there, so an epoch costs no host-to-device transfer.

For a fast pass that exercises every cell on a laptop CPU, set `PYVAE_SMOKE=1`
before starting Jupyter, or pass `smoke` as a parameter. The numbers will not
match the sweep, and the notebook says so where it matters.

Snakemake also runs this notebook through papermill and keeps the executed copy
at `outputs/01_kang_showcase.executed.ipynb`, with `epochs` and `seed` injected
from `config.yaml`.

In [ ]:
# Parameters. papermill injects overrides in a cell directly below this one, so
# the values here are what an interactive run uses. They match the sweep, which is
# why running the notebook untouched reproduces 04_fidelity_e250.csv row
# `notebook_both` rather than approximating it.
smoke = False
epochs = 250
seed = 42


In [ ]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import torch
from sklearn.model_selection import train_test_split

from pyvae import (
    InformedVAE,
    bayes_factor_da,
    build_model_config,
    differential_expression,
    integrated_gradients,
    load_kang,
    pathway_activity,
    pathway_unit_fidelity,
    pseudobulk_paired_test,
    set_all_seeds,
    swap_condition,
    train_ivae_modern,
)

# SMOKE trades fidelity for speed so every cell can be exercised on a laptop.
# It is not a smaller version of the experiment; it is a syntax check with numbers
# attached, and any figure produced under it should be thrown away. Set it through
# the `smoke` parameter above, or PYVAE_SMOKE=1 for an interactive session.
SMOKE = bool(smoke) or os.environ.get("PYVAE_SMOKE", "0") == "1"

# Every path below is relative to the repository root, matching the defaults the
# sweep scripts use. Jupyter and nbconvert both start the kernel in the notebook's
# own directory, so normalise before touching the filesystem. Skipping this is not
# a harmless error: load_kang downloads the dataset when it cannot find it, so a
# wrong working directory silently pulls 38 MB into
# experiments/kang/experiments/kang/data and the notebook appears to work.
_here = Path.cwd()
for _root in (_here, *_here.parents):
    if (_root / "pyvae" / "__init__.py").exists():
        os.chdir(_root)
        break
else:
    raise RuntimeError(f"could not find the pyvae repository root from {_here}")
print(f"working directory  {Path.cwd()}")

SEED = int(seed)
DATA_FOLDER = "experiments/kang/data"
GMT = "experiments/kang/resources/c2.cp.reactome.v7.5.1.symbols.gmt"
OUT = Path("experiments/kang/outputs")

# Exactly the configuration the sweep selected, so this run reproduces
# 04_fidelity_e250.csv row `notebook_both` rather than approximating it.
ARCH = dict(
    normalize="batch",
    informed_decoder=True,
    nonneg_encoder=True,
    standardize_input=True,
)
TRAIN = dict(
    epochs=int(epochs), patience=int(epochs), batch_size=128, lr=1e-3,
    weight_decay=1e-6, warmup_epochs=10, max_grad_norm=1.0,
)
N_GENES = None          # None means the full panel
TARGET_SUM = 1e4
L2_LAMBDA = 1e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if SMOKE:
    TRAIN |= dict(epochs=4, patience=4)
    N_GENES = 2000
    DEVICE = "cpu"

FIGS = OUT / ("figs_smoke" if SMOKE else "figs")
FIGS.mkdir(parents=True, exist_ok=True)

FOCUS = "REACTOME_INTERFERON_ALPHA_BETA_SIGNALING"
PARENT = "REACTOME_INTERFERON_SIGNALING"

mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "legend.frameon": False,
})

set_all_seeds(SEED)
print(f"torch {torch.__version__} | device {DEVICE} | smoke={SMOKE}")

## 1. Data

`load_kang` returns an AnnData whose `X` is log1p CP10K expression and whose
`counts` layer keeps the raw integers. Both are needed: the encoder reads the
normalised matrix because log1p expression is roughly symmetric and bounded enough
for a linear layer, while the decoder's negative-binomial likelihood is defined on
counts and would be meaningless on normalised values.

In [ ]:
adata = load_kang(
    data_folder=DATA_FOLDER,
    normalize=True,
    n_genes=N_GENES,
    return_path=False,
    target_sum=TARGET_SUM,
)
print(f"cells x genes      {adata.n_obs:,} x {adata.n_vars:,}")
print(f"conditions         {dict(adata.obs['condition'].value_counts())}")
print(f"cell types         {adata.obs['cell_type'].nunique()}")
print(f"donors             {adata.obs['replicate'].nunique()}")
print()
print(f"X      log1p CP10K, range [{float(adata.X.min()):.2f}, {float(adata.X.max()):.2f}]")
c = adata.layers["counts"]
c = c.toarray() if hasattr(c, "toarray") else np.asarray(c)
print(f"counts raw integers, range [{c.min():.0f}, {c.max():.0f}]")
del c

## 2. The Reactome map, and what it actually constrains

`build_model_config` reads the MSigDB Reactome GMT and returns a gene-by-pathway
binary matrix, intersected with the genes present in the data. Only genes appearing
in at least one pathway survive, so the model sees a smaller panel than the dataset
carries.

> **The point that is easy to get wrong.** The mask fixes each unit's *support*: which
> genes it may read. It says nothing about the *function*: the sign, the magnitude
> and therefore the meaning of the weights all stay free. A unit masked to the
> interferon gene set can still learn negative weights on all of them and fire
> *downward* when interferon signalling goes up. It would reconstruct perfectly well
> and be perfectly misleading. Section 5 measures how often that happens, and it is
> the reason the encoder needs more than a mask.

In [ ]:
x_trans = adata.to_df()
config = build_model_config(
    genes=x_trans.columns,
    model_kind="ivae_reactome",
    resources_dir=GMT,
)
adj_df = config.model_layer[0]
gene_names = list(config.input_genes)
pathway_names = list(adj_df.columns)

x_trans = x_trans[gene_names]
counts_df = adata.to_df(layer="counts")[gene_names]

sizes = adj_df.sum(axis=0)
print(f"adjacency          {len(gene_names):,} genes x {len(pathway_names):,} pathways")
print(f"genes dropped      {adata.n_vars - len(gene_names):,} of {adata.n_vars:,} "
      f"are in no Reactome pathway")
print(f"density            {adj_df.values.mean():.4%} of entries are 1")
print(f"pathway size       median {sizes.median():.0f}, "
      f"IQR {sizes.quantile(.25):.0f} to {sizes.quantile(.75):.0f}, max {sizes.max():.0f}")
print(f"genes per pathway  {(adj_df.sum(axis=1) > 0).sum():,} genes used, "
      f"a gene sits in {adj_df.sum(axis=1).mean():.1f} pathways on average")
print()
for p in (FOCUS, PARENT):
    print(f"{'present' if p in pathway_names else 'ABSENT ':<8} {p} "
          f"({int(sizes[p]) if p in pathway_names else 0} genes)")

### Overlapping gene sets set a floor on what any ranking can resolve

Reactome pathways share genes, so no method can separate two pathways that differ by
one member. This matters for interpreting Section 7: a pathway can rank highly
because it genuinely responds, or because it contains one massively induced gene
that it shares with the real answer. The overlap structure below is what makes that
ambiguity unavoidable rather than a defect of the model.

In [ ]:
focus_genes = set(adj_df.index[adj_df[FOCUS].values > 0]) if FOCUS in adj_df else set()
if focus_genes:
    shared = adj_df.loc[sorted(focus_genes)].sum(axis=0)
    shared = shared[shared > 0].drop(index=[FOCUS], errors="ignore").sort_values(ascending=False)
    ov = pd.DataFrame({
        "shared_with_focus": shared.astype(int),
        "pathway_size": sizes[shared.index].astype(int),
    })
    ov["fraction_of_pathway"] = ov.shared_with_focus / ov.pathway_size
    print(f"{FOCUS} has {len(focus_genes)} genes, shared with "
          f"{len(ov):,} other pathways\n")
    print("Most overlapping:")
    print(ov.head(8).to_string())
    print("\nPathways sharing exactly one gene with it: "
          f"{(ov.shared_with_focus == 1).sum():,}")

## 3. Covariates and the split

Condition and cell type enter as one-hot covariates through both encoder and
decoder. Conditioning on cell type matters here because the PBMC composition is
shared across conditions: without it, a unit can score well by separating monocytes
from T cells rather than by responding to interferon.

The train/validation split is stratified on condition crossed with cell type, so
every stratum appears on both sides in proportion.

In [ ]:
cov = pd.get_dummies(adata.obs[["condition", "cell_type"]], dtype=np.float32)
CTRL, STIM = "condition_control", "condition_stimulated"
assert {CTRL, STIM} <= set(cov.columns), sorted(cov.columns)

strat_full = adata.obs["condition"].astype(str) + "_" + adata.obs["cell_type"].astype(str)
strat = strat_full if strat_full.value_counts().min() >= 2 else adata.obs["condition"].astype(str)
idx_tr, idx_va = train_test_split(
    np.arange(len(adata)), test_size=0.2, stratify=strat, random_state=SEED,
)

data = {
    "x_train": x_trans.iloc[idx_tr].astype(np.float32),
    "x_val": x_trans.iloc[idx_va].astype(np.float32),
    "counts_train": counts_df.iloc[idx_tr].astype(np.float32),
    "counts_val": counts_df.iloc[idx_va].astype(np.float32),
    "cov_train": cov.iloc[idx_tr],
    "cov_val": cov.iloc[idx_va],
}
n_cov = cov.shape[1]
print(f"covariates         {n_cov} columns: {list(cov.columns)}")
print(f"train / val        {len(idx_tr):,} / {len(idx_va):,} cells")
print(f"val composition    "
      f"{int((data['cov_val'][CTRL] > .5).sum()):,} control, "
      f"{int((data['cov_val'][STIM] > .5).sum()):,} stimulated")

## 4. The model, and why this configuration

Four switches are on. The sweep in `02_ablation_merged.csv` and
`04_fidelity_e250.csv` is what chose them, and each earns its place for a different
reason:

| switch | what it does | what the sweep shows |
| --- | --- | --- |
| `normalize="batch"` | batch norm before the encoder's tanh | peak unit saturation falls from 0.996 to 0.089, and the count of units more than half saturated from 8 to 0. Layer norm makes it worse, 18 units. |
| `informed_decoder=True` | masks the decoder symmetrically | the three best validation losses in the eight-config grid are exactly the three informed-decoder configs, and counterfactual correlation rises from 0.30 to 0.53 |
| `nonneg_encoder=True` | softplus reparameterisation of encoder weights | the fraction of inverted units goes from 0.518 to exactly 0.000, and costs 0.03 nats |
| `standardize_input=True` | non-affine batch norm on the input | lifts median fidelity from 0.66 to 0.79 on top of nonnegativity; on its own it does nothing |

Initialisation stays at xavier. The sweep's `init="fan_in"` costs 14 nats alone and
returns 0.35 with everything else on, which is inside noise.

**The configuration with the lowest validation loss is not this one.** `all`
(fan-in, batch norm, informed decoder, no encoder constraints) reaches 1307.76
against 1308.90 here. It also has a sign agreement of 0.472, a coin flip, so its
units are as likely to encode their pathway upside-down as not. Spending 1.14 nats
to make that structurally impossible is the trade this notebook argues for, and it
is only visible because the sweep measured both.

In [ ]:
n_pathways = adj_df.shape[1]
latent_dim = n_pathways // 2
adj_tensor = torch.tensor(adj_df.values, dtype=torch.float32)

set_all_seeds(SEED)
model = InformedVAE(
    adj=adj_tensor,
    latent_dim=latent_dim,
    seed=SEED,
    l2_lambda=L2_LAMBDA,
    likelihood="nb",
    n_cov=n_cov,
    **ARCH,
)

tot = sum(p.numel() for p in model.parameters())
enc = sum(p.numel() for n, p in model.named_parameters() if "encoder" in n)
dec = sum(p.numel() for n, p in model.named_parameters() if "decoder" in n)
dense_first = len(gene_names) * n_pathways
print(f"pathway units      {n_pathways:,}    latent dim {latent_dim:,}")
print(f"parameters         {tot:,} total ({enc:,} encoder, {dec:,} decoder)")
print(f"mask saving        first layer holds {int(adj_tensor.sum()):,} live weights "
      f"of {dense_first:,} dense, a {1 - adj_tensor.sum()/dense_first:.1%} reduction")
print()
print("architecture:", {k: v for k, v in ARCH.items()})

## 5. Training

Cosine annealing over the full budget, with a KL warmup over the first 10 epochs.

> **Read the loss curve carefully.** `CosineAnnealingLR(T_max=epochs)` ties the
> schedule to the budget, so the learning rate reaches zero exactly at the last
> epoch and the best validation loss almost always lands there. That means
> `best_epoch == epochs` is the design working, not a sign of undertraining, and it
> means two budgets are two different experiments rather than one checkpointed
> twice. Compare final validation loss within a budget; never compare `best_epoch`
> across budgets.

In [ ]:
t0 = time.time()
model, history = train_ivae_modern(
    model,
    x_train=data["x_train"], x_val=data["x_val"],
    x_counts_train=data["counts_train"], x_counts_val=data["counts_val"],
    cov_train=data["cov_train"], cov_val=data["cov_val"],
    device=DEVICE,
    **TRAIN,
)
train_time = time.time() - t0
hist = pd.DataFrame(history)

# Everything downstream builds tensors on DEV. Read it off the model rather than
# reusing the DEVICE constant: the model is the authority on where it lives, and a
# mismatch surfaces as "Expected all tensors to be on the same device", which a
# CPU-only test run can never reproduce because there is only one device to be on.
DEV = next(model.parameters()).device
best_val, best_epoch = hist["val"].min(), int(hist["val"].idxmin()) + 1
print(f"\ntrained {len(hist)} epochs in {train_time/60:.1f} min on {DEVICE}")
print(f"best val loss      {best_val:.4f} at epoch {best_epoch} of {len(hist)}")
print(f"final val loss     {hist['val'].iloc[-1]:.4f}")
if not SMOKE:
    print(f"\nsweep reference    1308.8978  (04_fidelity_e250.csv, config notebook_both)")
    print(f"difference         {best_val - 1308.8978:+.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))

ax = axes[0]
ax.plot(hist.index + 1, hist["train"], label="train", lw=1.2)
ax.plot(hist.index + 1, hist["val"], label="validation", lw=1.2)
ax.axvline(best_epoch, color="0.4", ls=":", lw=1)
ax.set(xlabel="epoch", ylabel="negative ELBO", title="Loss")
ax.legend()

ax = axes[1]
ax.plot(hist.index + 1, hist["recon"], color="C2", lw=1.2, label="reconstruction")
ax.plot(hist.index + 1, hist["val"] - hist["recon"], color="C3", lw=1.2, label="val minus recon")
ax.set(xlabel="epoch", title="Reconstruction against the rest")
ax.legend()

ax = axes[2]
ax.plot(hist.index + 1, hist["lr"], color="C4", lw=1.2, label="learning rate")
ax.set(xlabel="epoch", title="Cosine schedule and KL warmup", yscale="log")
ax2 = ax.twinx()
ax2.plot(hist.index + 1, hist["beta"], color="C1", lw=1.2, ls="--", label="beta")
ax2.set_ylabel("beta"); ax2.grid(False)
lines = ax.get_lines() + ax2.get_lines()
ax.legend(lines, [l.get_label() for l in lines], loc="center right")

fig.suptitle(f"Training, {len(hist)} epochs, best val {best_val:.2f} at epoch {best_epoch}", y=1.03)
fig.tight_layout(); fig.savefig(FIGS / "01_training.png"); plt.show()

### Where batch normalisation earns its place

Each pathway unit is a tanh of a masked linear map. Without normalisation the
pre-activations drift far enough that tanh saturates, the gradient vanishes and the
unit stops responding to its genes at all. A single mean saturation figure hides
this completely: the sweep's baseline reports a mean of 0.005 while its worst unit
sits at 0.996 and eight units are more than half saturated.

The histogram below is per unit, so a saturated tail is visible rather than averaged
away.

In [ ]:
model.eval()
with torch.no_grad():
    xb = torch.tensor(data["x_val"].values, dtype=torch.float32, device=DEV)
    cb = torch.tensor(data["cov_val"].values, dtype=torch.float32, device=DEV)
    _, _, h_val = model.encode(xb, cb)
h_val_np = h_val.detach().cpu().numpy()

# Saturation: the fraction of cells for which a unit sits in the flat tail of tanh.
sat_per_unit = (np.abs(h_val_np) > 0.99).mean(axis=0)
sat_summary = dict(
    mean=float(sat_per_unit.mean()),
    max=float(sat_per_unit.max()),
    over_50pct=int((sat_per_unit > 0.5).sum()),
)
print(f"per-unit saturation   mean {sat_summary['mean']:.4f}   "
      f"max {sat_summary['max']:.4f}   units over 50% saturated {sat_summary['over_50pct']}")
if not SMOKE:
    print("sweep, this config    max 0.084, 0 units over 50%  (02_ablation_merged.csv, norm+informed_dec)")
    print("sweep, no batch norm  max 0.996, 8 units over 50%  (02_ablation_merged.csv, baseline)")

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.hist(sat_per_unit, bins=60, color="C0", edgecolor="white", lw=.4)
ax.axvline(0.5, color="C3", ls="--", lw=1, label="more than half the cells saturated")
ax.set(xlabel="fraction of cells with |h| > 0.99", ylabel="pathway units",
       title="Unit saturation, per unit rather than averaged", yscale="log")
ax.legend()
fig.tight_layout(); fig.savefig(FIGS / "02_saturation.png"); plt.show()

## 6. Do the units track their pathways?

This is the question the whole architecture exists to answer. `pathway_unit_fidelity`
correlates each unit's activation across cells against the mean z-scored expression
of that unit's own member genes. A unit whose correlation is near +1 represents its
pathway. Near zero it represents nothing. **Negative and it represents its pathway
upside-down**, which is worse than useless, because every ranking derived from it
carries the wrong sign while looking perfectly healthy.

Pathways with fewer than ten measured members are not tested, and they report `NaN`
rather than zero. Keep that distinction when you summarise: an untested pathway has
no sign, so counting it as "not inverted" would dilute the fraction with pathways
that were never measured. Every rate below is over the tested units only, and the
untested count is reported alongside it.

In [ ]:
fid = pathway_unit_fidelity(
    model,
    x=data["x_val"].values,
    adj=adj_df,
    cov=data["cov_val"].values,
    min_genes=10,
)
tested = fid[fid["corr"].notna()]
n_inv = int((tested["corr"] < 0).sum())

summary = {
    "pathways": len(fid),
    "tested": len(tested),
    "median_abs_corr": float(tested["abs_corr"].median()),
    "p90_abs_corr": float(tested["abs_corr"].quantile(0.90)),
    "fraction_below_02": float((tested["abs_corr"] < 0.2).mean()),
    "fraction_inverted": n_inv / len(tested),
}
for k, v in summary.items():
    print(f"  {k:<20} {v:.4f}" if isinstance(v, float) else f"  {k:<20} {v:,}")
print(f"\n  untested (fewer than 10 measured members): {len(fid) - len(tested):,}")
if not SMOKE:
    print("\n  sweep, this config    median 0.7914, below 0.2 0.0000, inverted 0.0000")
    print("  sweep, no constraints median 0.1251, below 0.2 0.6922, inverted 0.5183")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8),
                         gridspec_kw={"width_ratios": [1.5, 1]})

ax = axes[0]
ax.hist(tested["corr"], bins=70, color="C0", edgecolor="white", lw=.4)
ax.axvline(0, color="k", lw=1)
ax.axvspan(-1, 0, color="C3", alpha=0.07)
ax.axvline(0.2, color="C1", ls="--", lw=1)
ax.axvline(-0.2, color="C1", ls="--", lw=1)
ax.set(xlabel="corr(unit, mean z-scored member genes)", ylabel="pathway units",
       xlim=(-1, 1),
       title=f"Fidelity: {len(tested):,} tested units, {n_inv} inverted")
ax.text(-0.55, ax.get_ylim()[1] * .75, "inverted\n(pathway encoded\nupside-down)",
        ha="center", color="C3", fontsize=9)

ax = axes[1]
if not SMOKE:
    labels = ["no constraints", "nonneg only", "this model"]
    vals = [0.5183, 0.0000, summary["fraction_inverted"]]
    med = [0.1251, 0.6646, summary["median_abs_corr"]]
    xp = np.arange(3)
    ax.bar(xp - 0.19, vals, width=0.36, color="C3", label="fraction inverted")
    ax.bar(xp + 0.19, med, width=0.36, color="C0", label="median |corr|")
    ax.set_xticks(xp); ax.set_xticklabels(labels, fontsize=9)
    ax.set(ylim=(0, 1), title="Nonnegativity is the operative constraint")
    ax.legend(fontsize=9)
    for x, v in zip(xp - 0.19, vals):
        ax.text(x, v + .02, f"{v:.3f}", ha="center", fontsize=8, color="C3")
else:
    ax.text(.5, .5, "comparison needs the full run", ha="center", va="center")
    ax.set_axis_off()

fig.tight_layout(); fig.savefig(FIGS / "03_fidelity.png"); plt.show()

### Why nonnegativity makes inversion impossible rather than unlikely

`nonneg_encoder=True` reparameterises each masked weight as `w = softplus(theta) * mask`.
Since `softplus` is strictly positive, the encoder cannot represent a negative weight
at all. A unit therefore cannot anti-correlate with its own gene set, whatever the
data does. That is why the measured inverted fraction is exactly `0.0000` and not
merely small: the failure mode has been removed from the hypothesis space rather
than trained away.

The sweep says this holds across two architectural bases and two epoch budgets,
sixteen runs, with no exception, and that it costs 0.03 nats of validation loss.

It also shows what the constraint does **not** fix. Sign agreement, measured on the
decoder rather than the encoder, stays between 0.472 and 0.516 in all eight
configurations of `02_ablation_merged.csv`. No normalisation, initialisation or
decoder choice moves it off a coin flip. Fixing the decoder side needs the same
reparameterisation applied there, which none of the configurations implement, and
that is the natural next step for this line of work.

## 7. Where the interferon signal sits across cells

Embedding `h` directly does not work well, and the reason is worth stating rather
than hiding. The interferon response is by far the largest source of variation in
pathway space, so PCA on `h` puts nearly all of the leading variance on one
gradient, and UMAP renders a dominant monotone gradient as a horseshoe: two arms
running low to high response, with cell type smeared along them. That is the Guttman
arch, a property of the geometry rather than a defect in the model, and no amount of
filtering removes it because the gradient is genuinely there.

So embed on expression instead, with the standard pipeline, and overlay the model's
read-out on it. This is the stronger demonstration anyway. Colouring a pathway unit
onto an embedding the model never saw asks whether the unit agrees with structure
derived independently of it. Colouring it onto an embedding built from the model's
own output could only ever agree with itself.

What to look for: cell types resolving into the usual islands, the interferon unit
dark in control cells and bright in stimulated ones within every island, and the
companion plot showing that the response is present in all cell types but far
stronger in monocytes.


In [ ]:
# Pathway activations for every cell. h is informed(x) alone: the covariates are
# concatenated only downstream, on the way to z.
with torch.no_grad():
    xa = torch.tensor(x_trans.values, dtype=torch.float32, device=DEV)
    ca = torch.tensor(cov.values, dtype=torch.float32, device=DEV)
    _, _, h_all = model.encode(xa, ca)
h_np = h_all.detach().cpu().numpy()
del xa, ca, h_all

# Reference embedding, built from expression by the standard pipeline and entirely
# independent of the model. Highly variable genes, scale, PCA, neighbours, UMAP.
n_hvg = min(2000, adata.n_vars)
hvg = sc.pp.highly_variable_genes(adata, n_top_genes=n_hvg, flavor="seurat", inplace=False)
Xh = adata[:, hvg["highly_variable"].values].X
Xh = Xh.toarray() if hasattr(Xh, "toarray") else np.asarray(Xh)
emb = ad.AnnData(X=Xh.copy(), obs=adata.obs.copy())
sc.pp.scale(emb, max_value=10)
sc.tl.pca(emb, n_comps=50, random_state=SEED)
sc.pp.neighbors(emb, n_neighbors=15, random_state=SEED)
sc.tl.umap(emb, random_state=SEED)

# Overlay the model's read-out onto that frame.
UNIT = "interferon a/b unit"
emb.obs[UNIT] = h_np[:, pathway_names.index(FOCUS)] if FOCUS in pathway_names else np.nan

print(f"h            {h_np.shape[0]:,} cells x {h_np.shape[1]:,} pathway units, "
      f"range [{h_np.min():.3f}, {h_np.max():.3f}]")
print(f"embedding    {emb.n_obs:,} cells on {n_hvg:,} highly variable genes, "
      f"50 PCs carrying {emb.uns['pca']['variance_ratio'].sum():.1%} of variance")
print(f"{UNIT}  range [{emb.obs[UNIT].min():.3f}, {emb.obs[UNIT].max():.3f}]")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.6))
for ax, key, title in [
    (axes[0], "cell_type", "Cell type"),
    (axes[1], "condition", "Condition"),
    (axes[2], UNIT, "Interferon alpha/beta unit"),
]:
    sc.pl.umap(emb, color=key, ax=ax, show=False, frameon=False, title=title,
               size=5, legend_fontsize=8,
               cmap="magma" if key == UNIT else None)
fig.suptitle("Expression-space embedding, with the model's pathway unit overlaid", y=1.02)
fig.tight_layout(); fig.savefig(FIGS / "04_umap.png"); plt.show()


In [ ]:
# Companion: how strong is the response, and in which cells?
# Order by the thing the figure is about: how far the unit moves between
# conditions. Ordering by overall median would put the plot and the table below
# in different orders whenever a cell type is high in both conditions.
med = emb.obs.groupby(["cell_type", "condition"], observed=True)[UNIT].median().unstack()
shift = (med["stimulated"] - med["control"]).sort_values(ascending=False)
order = shift.index.tolist()
fig, ax = plt.subplots(figsize=(10, 4.2))
for k, cond in enumerate(["control", "stimulated"]):
    parts = []
    for ct in order:
        v = emb.obs.loc[(emb.obs.cell_type == ct) & (emb.obs.condition == cond), UNIT].values
        parts.append(v if len(v) > 1 else np.array([np.nan, np.nan]))
    pos = np.arange(len(order)) + (k - 0.5) * 0.38
    vp = ax.violinplot(parts, positions=pos, widths=0.34, showmedians=True,
                       showextrema=False)
    for b in vp["bodies"]:
        b.set_facecolor(["C0", "C3"][k]); b.set_alpha(0.75); b.set_edgecolor("none")
    vp["cmedians"].set_color("k"); vp["cmedians"].set_linewidth(1)
ax.set_xticks(np.arange(len(order)))
ax.set_xticklabels([c.replace(" cells", "") for c in order], fontsize=9)
ax.set_ylabel("unit activation")
ax.set_title("Interferon alpha/beta unit by cell type, control against stimulated")
ax.legend(handles=[plt.Line2D([], [], color=c, lw=6, label=l)
                   for c, l in [("C0", "control"), ("C3", "stimulated")]], fontsize=9)

tab = pd.DataFrame({
    "control": med["control"], "stimulated": med["stimulated"], "shift": shift,
    "n_cells": emb.obs.cell_type.value_counts(),
}).loc[order]
print("Median unit activation by cell type, and the shift between conditions:")
print(tab.to_string(float_format=lambda x: f"{x:.3f}"))

fig.tight_layout(); fig.savefig(FIGS / "04b_interferon_by_celltype.png"); plt.show()


## 8. Ranking pathways

Two rankings are available and they answer different questions.

The **encoder-side** test asks whether a unit's own activation shifts between
conditions. It is self-contained, which is its weakness: any unit whose activation
rises scores well, including a unit that rises only because it happens to contain one
massively induced gene shared with the real answer.

The **decoder-side** test is competitive. It samples posterior differential
expression gene by gene, then asks whether a pathway's genes are shifted *relative to
all other genes*. A pathway carrying one induced gene among forty unmoved ones does
not survive that comparison. This is the ranking to trust, and it is the one the
sweep scores.

In [ ]:
ctrl_mask = data["cov_val"][CTRL].values > 0.5
stim_mask = data["cov_val"][STIM].values > 0.5

de = differential_expression(
    model,
    x_a=data["x_val"].iloc[stim_mask], x_b=data["x_val"].iloc[ctrl_mask],
    cov_a=data["cov_val"].iloc[stim_mask], cov_b=data["cov_val"].iloc[ctrl_mask],
    n_samples=25 if not SMOKE else 5,
    n_pairs=2000 if not SMOKE else 200,
    gene_names=gene_names,
    seed=SEED,
)
pa = pathway_activity(de, adj_df, statistic="lfc_mean", min_genes=10)
pa = pa.sort_values("z_competitive", ascending=False)

rank_of = {p: i + 1 for i, p in enumerate(pa.index)}
print(f"posterior DE   {len(de):,} genes    pathways scored {len(pa):,}\n")
for p in (PARENT, FOCUS):
    if p in rank_of:
        r = pa.loc[p]
        print(f"  rank {rank_of[p]:>4}  z={r['z_competitive']:+.3f}  "
              f"auc={r['auc']:.3f}  q={r['qvalue']:.2e}  {p}")
        # Whether the pathway's score rests on the whole member set or on one
        # gene. A share near 1/n_tested means every member contributes about
        # equally; near 1.0 means the hit is really a gene-level finding.
        print(f"             top member {r['top_gene']} carries "
              f"{r['top_gene_share']:.1%} of the members' |lfc|, "
              f"against {1 / r['n_tested']:.1%} if they contributed equally")
print(f"\nTop 12 by competitive z:")
print(pa.head(12)[["n_genes", "z_competitive", "auc", "top_gene", "top_gene_share"]].to_string())
if not SMOKE:
    print(f"\nsweep reference  z = 9.1677 for {FOCUS}  (04_fidelity_e250.csv)")

In [ ]:
top = pa.head(18).iloc[::-1]
colors = ["C3" if p in (FOCUS, PARENT) else "0.65" for p in top.index]
labels = [p.replace("REACTOME_", "").replace("_", " ").lower()[:52] for p in top.index]

fig, ax = plt.subplots(figsize=(8.5, 6))
ax.barh(np.arange(len(top)), top["z_competitive"], color=colors)
ax.set_yticks(np.arange(len(top))); ax.set_yticklabels(labels, fontsize=8)
ax.set(xlabel="competitive z (pathway genes against all other genes)",
       title="Decoder-side competitive ranking, stimulated against control")
ax.axvline(0, color="k", lw=.8)
for i, (v, p) in enumerate(zip(top["z_competitive"], top.index)):
    if p in (FOCUS, PARENT):
        ax.text(v + .1, i, "known answer", va="center", fontsize=8, color="C3")
fig.tight_layout(); fig.savefig(FIGS / "05_competitive_ranking.png"); plt.show()

### The two rankings side by side

The encoder-side Bayes factor keeps DNA-repair and viral-response pathways near the
top. That is not the encoder misbehaving: Section 2 showed those gene sets each share
a single gene with the interferon set, and that gene is among the most induced in the
dataset. A self-contained test has no way to discount it. The competitive test does,
and pushes those pathways down by hundreds of ranks.

In [ ]:
bf = bayes_factor_da(
    model,
    x_a=torch.tensor(data["x_val"].iloc[stim_mask].values, dtype=torch.float32, device=DEV),
    x_b=torch.tensor(data["x_val"].iloc[ctrl_mask].values, dtype=torch.float32, device=DEV),
    cov_a=torch.tensor(data["cov_val"].iloc[stim_mask].values, dtype=torch.float32, device=DEV),
    cov_b=torch.tensor(data["cov_val"].iloc[ctrl_mask].values, dtype=torch.float32, device=DEV),
    n_pairs=2000 if not SMOKE else 200,
    pathway_names=pathway_names,
    seed=SEED,
)
# bayes_factor_da returns pathway names in the index, already sorted by |bf|.
# Re-sort on signed bf so "top" means most up-regulated rather than most extreme.
bf = bf.sort_values("bf", ascending=False)
bf_rank = {p: i + 1 for i, p in enumerate(bf.index)}

rows = []
for p in list(bf.index)[:10]:
    rows.append({
        "pathway": p.replace("REACTOME_", "")[:46],
        "encoder_rank": bf_rank.get(p),
        "competitive_rank": rank_of.get(p, None),
        "shared_with_focus": int(adj_df.loc[sorted(focus_genes), p].sum()) if focus_genes and p in adj_df else None,
        "pathway_size": int(sizes[p]) if p in sizes else None,
    })
cmp_df = pd.DataFrame(rows)
print("Top 10 by encoder-side Bayes factor, and where the competitive test puts them:\n")
print(cmp_df.to_string(index=False))

## 9. Which genes drive the top pathway

Integrated gradients attributes a unit's activation back to individual input genes.
For an informed encoder every attribution falls on the pathway's own members by
construction, so that check is a sanity test rather than a finding, and the real
question is how the weight is distributed across them.

Neither extreme would be good news. All the weight on one gene would mean the unit is
a single-gene detector wearing a pathway's name. Perfectly uniform weight would mean
it ignores which genes actually responded. Expect something in between, dominated by
the genes that are genuinely most induced, and read the concentration figures below
rather than assuming either.

These attributions measure the unit's *sensitivity*, which is a different quantity
from the pathway's *differential expression*. Section 7 reported the second one:
`top_gene_share` says how much of the members' summed absolute log fold change the
largest single member carries. A pathway can respond broadly while the unit reads
mostly a handful of its genes, so the two numbers need not agree, and on this model
they do not. Both are printed below so the gap is visible rather than implied.

In [ ]:
target = FOCUS if FOCUS in pathway_names else pa.index[0]
pidx = pathway_names.index(target)
n_ig = min(256, stim_mask.sum())
sel = np.where(stim_mask)[0][:n_ig]

attr = integrated_gradients(
    model,
    x=torch.tensor(data["x_val"].iloc[sel].values, dtype=torch.float32, device=DEV),
    pathway_idx=pidx,
    cov=torch.tensor(data["cov_val"].iloc[sel].values, dtype=torch.float32, device=DEV),
    steps=50 if not SMOKE else 8,
)
ga = attr.detach().cpu().numpy().mean(axis=0)
attr_df = (pd.DataFrame({"gene": gene_names, "attribution": ga})
           .assign(abs_attribution=lambda d: d.attribution.abs())
           .sort_values("abs_attribution", ascending=False).reset_index(drop=True))

members = set(adj_df.index[adj_df[target].values > 0])
in_set = attr_df.gene.isin(members)
print(f"target             {target}  ({len(members)} members)")
print(f"attribution mass   {attr_df.loc[in_set,'abs_attribution'].sum() / attr_df.abs_attribution.sum():.2%} "
      f"falls on member genes")
top_share = attr_df.abs_attribution.iloc[0] / attr_df.abs_attribution.sum()
print(f"concentration      top gene carries {top_share:.1%}; "
      f"top 5 carry {attr_df.abs_attribution.head(5).sum()/attr_df.abs_attribution.sum():.1%}")

# The decoder-side counterpart from Section 7. Sensitivity (what the unit reads)
# and differential expression (what the member genes did) are different quantities.
if target in pa.index and pd.notna(pa.loc[target, "top_gene_share"]):
    r = pa.loc[target]
    print(f"compare, decoder   {r['top_gene']} carries {r['top_gene_share']:.1%} of the "
          f"members' |lfc|, against {1 / r['n_tested']:.1%} if they contributed equally")
print()
print(attr_df.head(15).to_string(index=False))

In [ ]:
t = attr_df.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.barh(np.arange(len(t)), t.attribution,
        color=["C0" if g in members else "C3" for g in t.gene])
ax.set_yticks(np.arange(len(t))); ax.set_yticklabels(t.gene, fontsize=8)
ax.axvline(0, color="k", lw=.8)
ax.set(xlabel="mean integrated-gradients attribution",
       title=f"Gene attribution for {target.replace('REACTOME_','').replace('_',' ').lower()}")
fig.tight_layout(); fig.savefig(FIGS / "06_attribution.png"); plt.show()

## 10. Counterfactual prediction

Take control cells, hold their inferred latent biology fixed, flip only the condition
covariate, and decode. The model predicts what those same cells would have expressed
under stimulation.

**Both sides of the comparison must come from the decoder.** The obvious version,
subtracting the observed control expression from the predicted stimulated counts,
mixes a decoder output with a raw input, so every gene inherits the decoder's own
reconstruction bias. That shows up as a systematic negative offset on genes that did
not move and as roughly twofold shrinkage on the ones that did. Decoding the same
cells twice, once with the covariate unchanged and once swapped, cancels that bias
because it appears on both sides.

`02_ablation_merged.csv` measures this for all eight architectures, and it is the
metric that separates them most cleanly. The three configurations with an informed
decoder score 0.978 to 0.981; the five without one score 0.803 to 0.861, with no
overlap between the groups. Masking the decoder so each pathway unit writes only to
its own member genes is what buys counterfactual fidelity, and it is a different
knob from the one that fixes saturation.

This is the one read-out a naive gene-set mean cannot produce at all.

In [ ]:
n_cf = min(1000, int(ctrl_mask.sum()))
ci = np.where(ctrl_mask)[0][:n_cf]
x_c = data["x_val"].iloc[ci]
cnt_c = data["counts_val"].iloc[ci]
cov_c = data["cov_val"].iloc[ci]

x_ct = torch.tensor(x_c.values, dtype=torch.float32, device=DEV)
cov_ct = torch.tensor(cov_c.values, dtype=torch.float32, device=DEV)
lib_ct = torch.tensor(cnt_c.values.sum(axis=1, keepdims=True), dtype=torch.float32, device=DEV)
cov_to = swap_condition(cov_c, from_label="control", to_label="stimulated")
cov_tt = torch.tensor(cov_to.values, dtype=torch.float32, device=DEV)

with torch.no_grad():
    # Decode the same cells twice. The unswapped pass is the baseline, so the
    # decoder's reconstruction bias is present on both sides and cancels.
    pred_ctrl = model.predict_counterfactual(x_ct, lib_ct, cov_ct, cov_ct)
    pred_stim = model.predict_counterfactual(x_ct, lib_ct, cov_ct, cov_tt)

pred_shift = (torch.log1p(pred_stim).mean(0)
              - torch.log1p(pred_ctrl).mean(0)).detach().cpu().numpy()
real_shift = data["x_val"].iloc[stim_mask].values.mean(0) - x_c.values.mean(0)
cf_corr = float(np.corrcoef(pred_shift, real_shift)[0, 1])

# The biased form, kept only to show the size of the artefact it introduces.
biased_shift = (torch.log1p(pred_stim).mean(0) - x_ct.mean(0)).detach().cpu().numpy()
cf_corr_biased = float(np.corrcoef(biased_shift, real_shift)[0, 1])

# Calibration: correlation says the direction is right, slope says whether the
# magnitude is. They are different claims and the figure should show both.
slope, intercept = np.polyfit(real_shift, pred_shift, 1)

print(f"counterfactual cells   {n_cf:,} control cells pushed to stimulated")
print(f"shift correlation      {cf_corr:.4f}   (decoder baseline, both sides decoded)")
print(f"  same, biased form    {cf_corr_biased:.4f}   (predicted minus observed control)")
print(f"calibration slope      {slope:.3f}   (1.0 would be perfectly calibrated)")
print(f"intercept              {intercept:+.4f}   (0.0 means unchanged genes stay put)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.scatter(real_shift, pred_shift, s=4, alpha=.25, color="C0", lw=0)
lim = np.array([min(real_shift.min(), pred_shift.min()), max(real_shift.max(), pred_shift.max())])
ax.plot(lim, lim, color="0.4", ls="--", lw=1, label="identity")
ax.plot(lim, slope * lim + intercept, color="C1", lw=1.2,
        label=f"fit, slope {slope:.2f}")
mem = np.array([g in members for g in gene_names])
ax.scatter(real_shift[mem], pred_shift[mem], s=14, color="C3", lw=0,
           label=f"{target.replace('REACTOME_','').lower()[:28]} members")
ax.set(xlabel="observed shift (log1p, stimulated minus control)",
       ylabel="predicted shift", title=f"Counterfactual, r = {cf_corr:.3f}")
ax.legend(fontsize=8)

ax = axes[1]
k = 18
o = np.argsort(real_shift)[::-1][:k]
y = np.arange(k)
ax.barh(y - .2, real_shift[o], height=.4, color="0.55", label="observed")
ax.barh(y + .2, pred_shift[o], height=.4, color="C0", label="predicted")
ax.set_yticks(y); ax.set_yticklabels([gene_names[i] for i in o], fontsize=8)
ax.invert_yaxis()
ax.set(xlabel="log1p shift", title="Genes most induced in the data")
ax.legend(fontsize=9)

fig.tight_layout(); fig.savefig(FIGS / "07_counterfactual.png"); plt.show()

## 11. Honest statistics on the gene level

The cell-level tests above treat cells as independent replicates. They are not: cells
from one donor share that donor's genetics, handling and batch, so a cell-level test
inflates its own significance by counting the same information many times over
(Squair et al. 2021). The correct unit of replication is the donor, and Kang has
eight.

**Eight donors put a hard floor under the p-value.** A paired Wilcoxon on eight pairs
has 2^8 = 256 possible sign configurations, so the smallest two-sided p it can
produce is 2/256 = 0.0078, whatever the effect size. Every gene that moves in the
same direction in all eight donors lands on exactly that value and they tie, which
after Benjamini-Hochberg gives them all one shared q. Thousands of genes reaching
"significance" here is not thousands of independent discoveries; it is thousands of
genes that were consistent, ranked by nothing.

So read this table by effect size, not by q. The p-value tells you the direction was
consistent across donors, which is the claim the cell-level test could not support.
It cannot tell you which of the consistent genes moved most, because at eight donors
it has run out of resolution.


In [ ]:
pb = pseudobulk_paired_test(
    counts=adata.layers["counts"],
    donor_ids=adata.obs["replicate"].values,
    condition=adata.obs["condition"].values,
    label_a="stimulated", label_b="control",
    gene_names=adata.var_names.tolist(),
    min_donors=3, pseudocount=1.0, target_sum=1e6,
)
qcol = "qvalue" if "qvalue" in pb.columns else None
print(f"donors paired      {pb['n_donors_paired'].iloc[0]}")
print(f"genes tested       {len(pb):,}")
if qcol:
    for a in (0.05, 0.1):
        print(f"  q < {a}          {int((pb[qcol] < a).sum()):,} genes")
print()
print(pb.head(12).to_string())

## 12. How little data this needs

This is the one result taken directly from the sweep, because a single notebook run
cannot establish it: it needs the model retrained at seven training-set sizes with
five independent subsamples each, which is `03_sample_size_experiment.py`.

The comparison is against a model-free baseline: rank pathways by the log ratio of
group means, two lines of numpy and no VAE. If that baseline matches the model, the
model is not earning its complexity on this task.

On the full gene panel it does not match. The model puts interferon alpha/beta at
rank 1 in **every one of 25 runs**, down to 246 cells per condition, while the
baseline sits between 11 and 15. On a 5,000 highly-variable-gene panel both reach
rank 1, because selecting 5,000 variable genes has already done the feature selection
that makes interferon obvious. The panel width, not the cell count, is what separates
the two methods.

In [ ]:
ss_path = OUT / "03_sample_size_merged.csv"
if ss_path.exists():
    ss = pd.read_csv(ss_path)
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4), sharey=True)
    for ax, panel, title in [
        (axes[0], 0, "Full gene panel"),
        (axes[1], 5000, "5,000 highly variable genes"),
    ]:
        d = ss[ss.n_genes == panel]
        for method, color, marker in [("model", "C0", "o"), ("naive", "C3", "s")]:
            m = d[d.method == method].sort_values("n_train")
            ax.plot(m.n_train, m.interferon_ab_rank, marker=marker, color=color,
                    label=method, lw=1.4, ms=5)
        ax.set(xscale="log", yscale="log", xlabel="training cells",
               title=f"{title}\n(n_genes={panel or 'all'})")
        ax.axhline(1, color="0.5", ls=":", lw=1)
        ax.legend(title="ranking")
    axes[0].set_ylabel("rank of interferon alpha/beta\n(1 is correct, lower is better)")
    fig.suptitle("Where the pathway-constrained model beats a model-free baseline", y=1.04)
    fig.tight_layout(); fig.savefig(FIGS / "08_sample_size.png"); plt.show()

    print("Full panel, median over reps:")
    print(ss[ss.n_genes == 0]
          .pivot_table(index="n_train", columns="method", values="interferon_ab_rank")
          .to_string())
else:
    print(f"{ss_path} not found. Run the sweep first:")
    print("  cd experiments/kang && pixi run all")

## 13. What this model does and does not establish

**Established here, on one model, full panel.** Pathway units track their own gene
sets, with no unit encoding its pathway upside-down, and that property is structural
rather than fortunate. Batch normalisation keeps the units off tanh's flat tail. The
competitive decoder-side ranking recovers interferon signalling and its alpha/beta
child at the top, while the self-contained encoder-side ranking does not, for a
reason the gene-set overlap in Section 2 makes concrete. Gene attribution spreads
across a pathway's members rather than collapsing onto one. Condition conditioning
learned a treatment effect transferable to held-out cells.

**Established by the sweep, not here.** Which switches matter and what each costs
(`02_ablation_merged.csv`, eight configurations). That nonnegativity alone removes
inversion, on two bases and two epoch budgets (`04_fidelity_e250.csv`,
`04_fidelity_e100.csv`, sixteen runs). That the advantage over a model-free baseline
is real on the full panel and absent at 5,000 genes
(`03_sample_size_merged.csv`, 25 runs per panel). Compute cost, in
`benchmarks.csv`.

**Not established by either.** Whether any of this transfers beyond Kang, which is a
single dataset with a single strong perturbation and a known answer, which is exactly
why it is a benchmark and exactly why it cannot support a general claim. Whether
pathway units add per-cell discriminative power over a gene-set mean: the sweep says
they roughly tie, mean AUC difference 0.0023 with the model ahead on 54.6% of
pathways. The value is in the shared latent geometry, the counterfactual and the
named axes, none of which a gene-set mean provides, and none of which this notebook
measures against alternatives.

**The known gap.** Sign agreement on the decoder side sits at a coin flip in all
eight configurations of the sweep. The encoder's softplus reparameterisation has no
counterpart there. Applying it is the obvious next step, and until it is done, any
decoder-side interpretation carries an unresolved sign.

In [ ]:
run = {
    "device": DEVICE, "smoke": SMOKE, "seed": SEED,
    "cells": int(adata.n_obs), "genes_in_data": int(adata.n_vars),
    "genes_in_adjacency": len(gene_names), "pathways": n_pathways,
    "latent_dim": latent_dim, "n_cov": n_cov, "parameters": int(tot),
    "epochs": len(hist), "train_minutes": round(train_time / 60, 2),
    "best_val_loss": round(best_val, 4),
    "fidelity_median_abs_corr": round(summary["median_abs_corr"], 4),
    "fidelity_fraction_inverted": round(summary["fraction_inverted"], 4),
    "saturation_max": round(sat_summary["max"], 4),
    "saturation_units_over_50pct": sat_summary["over_50pct"],
    "competitive_z_focus": round(float(pa.loc[FOCUS, "z_competitive"]), 4) if FOCUS in pa.index else None,
    "competitive_rank_focus": rank_of.get(FOCUS),
    "counterfactual_corr": round(cf_corr, 4),
    "architecture": ARCH, "training": TRAIN,
}
dest = OUT / ("showcase_run_smoke.json" if SMOKE else "showcase_run.json")
dest.write_text(json.dumps(run, indent=2))
print(json.dumps(run, indent=2))
print(f"\nwrote {dest}")
print(f"figures in {FIGS}/")